In [5]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [6]:
df = pd.read_parquet("../source_data/gold_asset_failure_risk.parquet")
print(f"Shape: {df.shape}")

Shape: (261, 26)


In [7]:
# ── 2. Feature selection ─────────────────────────────────────────────────────
# Drop identifiers and existing ML output columns (we'll repopulate them)
DROP_COLS = [
    'hardware_pk', 'hardware_id', 'asset_name', 'uuid',
    'anomaly_score', 'is_anomaly',          # targets — we generate these
    'rule_based_risk_score',                 # kept only for validation later
    'source_year',                           # temporal ID, not a feature
]
FEATURE_COLS = [c for c in df.columns if c not in DROP_COLS + ['rule_based_risk_score']]

In [8]:
# Encode os_family (only categorical)
le = LabelEncoder()
df['os_family_encoded'] = le.fit_transform(df['os_family'].fillna('unknown'))
FEATURE_COLS = [c for c in FEATURE_COLS if c != 'os_family'] + ['os_family_encoded']

X = df[FEATURE_COLS].fillna(0)
print(f"Features used ({len(FEATURE_COLS)}): {FEATURE_COLS}")

Features used (18): ['device_age_years', 'bios_risk_level_encoded', 'worst_drive_health_encoded', 'total_drive_gb', 'drive_count', 'storage_device_count', 'high_risk_software_count', 'total_software_count', 'risk_software_ratio', 'memory_gb', 'cpu_cores', 'has_valid_ipv4', 'has_warranty', 'asset_value', 'incident_count', 'days_since_last_inventory', 'inventory_quality_tier_encoded', 'os_family_encoded']


In [9]:
# ── 3. Isolation Forest ──────────────────────────────────────────────────────
# contamination=0.05 → flags ~5% of assets as anomalies (~13 of 261)
# Tune this up (0.10) if your rule_based_risk_score suggests more risky assets
iso = IsolationForest(
    n_estimators=200,
    contamination=0.05,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
iso.fit(X)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",200
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary ` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",42
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary `... versionadded:: 0.21",False


In [10]:
# Raw decision function: more negative = more anomalous
raw_scores = iso.decision_function(X)          # range roughly (-0.5, 0.5)
predictions = iso.predict(X)                   # -1 = anomaly, 1 = normal

In [11]:
# Normalize score to [0, 1] where 1 = most anomalous
anomaly_score = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
is_anomaly = (predictions == -1).astype(int)

df['anomaly_score'] = np.round(anomaly_score, 4)
df['is_anomaly'] = is_anomaly

In [12]:
# ── 4. Validation: compare with rule_based_risk_score ────────────────────────
print("\n── Anomaly Detection Results ──")
print(f"  Total assets      : {len(df)}")
print(f"  Flagged anomalies : {is_anomaly.sum()} ({is_anomaly.mean()*100:.1f}%)")
print(f"  Avg anomaly_score : {anomaly_score.mean():.3f}")


── Anomaly Detection Results ──
  Total assets      : 261
  Flagged anomalies : 13 (5.0%)
  Avg anomaly_score : 0.153


In [13]:
# Do IF anomalies align with high rule_based_risk_score?
correlation = df[['rule_based_risk_score', 'anomaly_score']].corr().iloc[0, 1]
print(f"\n  Correlation(rule_based_risk_score, anomaly_score): {correlation:.3f}")
print("  ✅ Good alignment" if correlation > 0.4 else "  ⚠️  Low alignment — investigate")

print("\n── Top 10 Most Anomalous Assets ──")
top_anomalies = (
    df[['asset_name', 'anomaly_score', 'is_anomaly', 'rule_based_risk_score',
        'device_age_years', 'worst_drive_health_encoded', 'incident_count',
        'high_risk_software_count', 'days_since_last_inventory']]
    .sort_values('anomaly_score', ascending=False)
    .head(10)
)
print(top_anomalies.to_string(index=False))


  Correlation(rule_based_risk_score, anomaly_score): 0.277
  ⚠️  Low alignment — investigate

── Top 10 Most Anomalous Assets ──
 asset_name  anomaly_score  is_anomaly  rule_based_risk_score  device_age_years  worst_drive_health_encoded  incident_count  high_risk_software_count  days_since_last_inventory
    raja-pc         1.0000           1                      9                 0                           1               0                      57.0                       4362
 cp-chinoun         0.6568           1                     12                21                           1               2                      88.0                       4626
p-18050-047         0.5862           1                     14                18                           4               4                      62.0                       4320
p-02002-020         0.5775           1                     12                13                           1              11                       8.0             

In [14]:
# ── 5. Feature importance via mean anomaly score per feature quartile ─────────
# Simple proxy: compare mean feature values for anomalies vs normals
print("\n── Feature Differences (Anomaly vs Normal) ──")
numeric_features = [c for c in FEATURE_COLS if c != 'os_family_encoded']
diff = (
    df.groupby('is_anomaly')[numeric_features]
    .mean()
    .T
    .rename(columns={0: 'normal_mean', 1: 'anomaly_mean'})
)
diff['delta'] = diff['anomaly_mean'] - diff['normal_mean']
diff['pct_diff'] = (diff['delta'] / (diff['normal_mean'].abs() + 1e-6) * 100).round(1)
print(diff.sort_values('delta', ascending=False).to_string())


── Feature Differences (Anomaly vs Normal) ──
is_anomaly                      normal_mean  anomaly_mean       delta  pct_diff
total_drive_gb                   323.272379    660.871538  337.599159     104.4
total_software_count              73.782258    211.538462  137.756203     186.7
high_risk_software_count          10.362903     82.076923   71.714020     692.0
days_since_last_inventory       4186.782258   4215.461538   28.679280       0.7
drive_count                        5.185484      6.461538    1.276055      24.6
incident_count                     1.467742      2.692308    1.224566      83.4
memory_gb                          2.232500      3.353846    1.121346      50.2
cpu_cores                          1.008065      1.769231    0.761166      75.5
storage_device_count               3.907258      4.615385    0.708127      18.1
risk_software_ratio                0.105125      0.403338    0.298213     283.7
worst_drive_health_encoded         1.145161      1.230769    0.085608    

In [15]:
# ── 6. Export ─────────────────────────────────────────────────────────────────
df.to_parquet("../export_data/gold_asset_failure_risk.parquet", index=False)
print("\n✅ Parquet updated with anomaly_score + is_anomaly")



✅ Parquet updated with anomaly_score + is_anomaly


In [16]:
# Optional: save results summary
results = df[['hardware_pk', 'asset_name', 'anomaly_score', 'is_anomaly',
              'rule_based_risk_score']].copy()
results.to_csv("../export_data/asset_anomaly_results.csv", index=False)
print("✅ ../export_data/asset_anomaly_results.csv saved")

✅ ../export_data/asset_anomaly_results.csv saved


In [17]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

# Save model + label encoder
joblib.dump(iso, "../models/isolation_forest_asset_anomaly.pkl")
joblib.dump(le,  "../models/label_encoder_os_family.pkl")

print("✅ Models saved:")
print("   ../models/isolation_forest_asset_anomaly.pkl")
print("   ../models/label_encoder_os_family.pkl")

✅ Models saved:
   ../models/isolation_forest_asset_anomaly.pkl
   ../models/label_encoder_os_family.pkl
